# D · 원하람 — Logistic Regression + Sentence-BERT 문체 융합 (Kaggle)

- 코드: `haram` 브랜치의 `experiments/D_logistic_regression.py`, `experiments/D_fusion.py`
- train으로 학습, val로 비교. **test는 열지 않음.**
- Settings: 2~3절은 **Accelerator = None** 으로 충분(CPU 3분). 4절부터 **GPU T4 x2**, **Internet = On**.
- **Save Version → Save & Run All (Commit)** 으로 돌리면 브라우저를 닫아도 끝까지 돈다. 결과는 버전 Output 탭의 `D_results.zip`.

## 1. 저장소 받기·환경 기록

In [ ]:
%cd /kaggle/working
!rm -rf LLM-name-tagging
!git clone -b haram https://github.com/seoultech-Machine-Learning-26-2/LLM-name-tagging.git
%cd /kaggle/working/LLM-name-tagging
!git log -1 --oneline

import os
assert os.path.exists('experiments/D_logistic_regression.py'), 'haram 브랜치가 push 됐는지 확인'

import platform, sys, numpy, pandas, sklearn, scipy
print(platform.platform(), '| Python', sys.version.split()[0])
print('numpy', numpy.__version__, '| pandas', pandas.__version__,
      '| sklearn', sklearn.__version__, '| scipy', scipy.__version__)
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo 'GPU 없음'

## 1-1. 한글 폰트
설치 안 하면 그림 제목이 □ 로 나온다.

In [ ]:
!apt-get -qq install -y fonts-nanum > /dev/null 2>&1
!fc-cache -f > /dev/null 2>&1
!rm -rf ~/.cache/matplotlib
import matplotlib.font_manager as fm
print('NanumGothic:', any('Nanum' in f.name for f in fm.fontManager.ttflist))

## 2. Logistic Regression (D-1 ~ D-2)
문체 32개 C=0.1/1/10 → C 고정 후 문체/문자 TF-IDF/결합 비교 → 입력별 C 재튜닝.
Dummy 기준선 3종, 스케일링 없는 문체, 최고 설정의 seed 42/43/44 변동도 같이 기록. CPU 약 3분.

In [ ]:
!python -u -W ignore -m experiments.D_logistic_regression

## 3. 결과 보기
선택 기준은 `model_BA`. val이 클래스당 22개씩이라 BA와 accuracy가 같게 나온다.
`converged=False` 인 행은 최적화 실패일 수 있어 규제 효과로 해석하지 않는다.

In [ ]:
import pandas as pd
from IPython.display import Image, display as d
pd.set_option('display.width', 220)

res = pd.read_csv('results/course/D/logistic_regression/val_results.csv')
cols = ['run_id','feature_set','train_model_BA','model_BA','accuracy','macro_f1',
        'log_loss','ece','sibling_acc','company_BA','converged','n_iter','fit_seconds']
display(res[[c for c in cols if c in res.columns]].round(4))

### 3-1. train/val 차이
train만 높으면 과적합, 둘 다 낮으면 표현력 부족.

In [ ]:
lr = res[res.model == 'LogisticRegression'].copy()
lr['gap'] = lr.train_model_BA - lr.model_BA
display(lr[['config','train_model_BA','model_BA','gap']].round(4).sort_values('gap', ascending=False))
d(Image('results/course/D/logistic_regression/fig_config_comparison.png'))

### 3-2. 혼동행렬
행 정규화라 대각선이 재현율. 같은 개발사 모델이 인접해 있어 블록으로 뭉치면 형제 혼동.

In [ ]:
d(Image('results/course/D/logistic_regression/fig_confusion_val.png'))

### 3-3. 오분류 5개
확신이 높은 순. `text_head` 를 읽고 관찰한 문체를 보고서에 직접 적는다.

In [ ]:
err = pd.read_csv('results/course/D/logistic_regression/val_errors_seed42.csv')
for _, r in err.iterrows():
    print(f"[{r.id}] {r.true_model} -> {r.pred_model} "
          f"(p={r.pred_prob:.3f}, 정답 p={r.true_prob:.3f}, 같은 개발사={r.same_company})")
    print('   ', r.text_head[:200], '\n')

### 3-5. 보정·형제 혼동·모델별 재현율

In [ ]:
B = 'results/course/D/logistic_regression/'
print('확신도 구간별 (gap > 0 이면 과신)')
display(pd.read_csv(B+'val_calibration.csv'))
print('오분류 중 같은 개발사 비율')
display(pd.read_csv(B+'val_sibling_errors.csv'))
print('모델별 재현율')
display(pd.read_csv(B+'val_per_class.csv'))

### 3-4. 계수
문체 32개는 계수를 직접 읽을 수 있다. 상관된 특징이 있으면 계수가 나뉘어 작아 보일 수 있다.

In [ ]:
coef = pd.read_csv('results/course/D/logistic_regression/coefficients_style.csv')
display(coef.feature.value_counts().head(12).to_frame('상위 등장 모델 수'))
display(coef.sort_values('abs_coef', ascending=False).groupby('model').head(1)
            [['model','feature','coef']].sort_values('model').reset_index(drop=True))

## 4. Sentence-BERT 문체 융합 (D-3 ~ D-4)
GPU 필요. Settings를 GPU로 바꾸고 1절부터 다시 실행.
같은 분할·seed·학습 예산으로 의미만 / 문체만 / 융합 세 모델을 비교한다. 임베딩은 동결.

In [ ]:
import torch
assert torch.cuda.is_available(), 'Settings에서 Accelerator를 GPU로 바꾸고 1절부터 다시 실행'
print(torch.__version__, torch.cuda.get_device_name(0))
!pip -q install -U sentence-transformers

In [ ]:
# D_fusion.py 작성 후 주석 해제
# !python -u -W ignore -m experiments.D_fusion --encoder sentence-transformers/all-MiniLM-L6-v2

## 5. 결과 저장
`/kaggle/working/D_results.zip` 이 버전 Output에 남는다.

In [ ]:
!zip -qr /kaggle/working/D_results.zip results/course/D preds/val_D_* preds/test_D_* 2>/dev/null
!ls -lh /kaggle/working/D_results.zip
!unzip -l /kaggle/working/D_results.zip | tail -25

## 6. test 1회 (최종 평가 전용)
팀이 최종 모델 목록을 고정한 뒤에만 실행. `--final` 에 2절 출력의 최고 설정 이름을 넣는다.

In [ ]:
# !python -u -W ignore -m experiments.D_logistic_regression --final combined_C10.0